In [1]:
# TODO : tester retrieval_task et generation_task separement

In [2]:
import os
from datetime import datetime
from langfuse import get_client
from langchain_qdrant import QdrantVectorStore, RetrievalMode

from polylex_chatbot.env import load_project_env

env_path = load_project_env()

from experiment.rag import make_rag_task
from experiment.evaluators import *

from polylex_chatbot.config import EMBEDDING_MODEL_CONFIG, SPARSE_MODEL_CONFIG_FR, DB_DENSE_INDEX_CONFIG, DB_SPARSE_INDEX_CONFIG_FR

In [3]:
langfuse = get_client()
DATASET_NAME = "20250520_clean_dev_dataset"
dataset = langfuse.get_dataset(DATASET_NAME)
RUN_NAME = datetime.now().isoformat()

embeddings = EMBEDDING_MODEL_CONFIG

qdrant = QdrantVectorStore.from_existing_collection(
    url=os.getenv("QDRANT_URL"),
    embedding=embeddings,
    sparse_embedding=SPARSE_MODEL_CONFIG_FR,
    collection_name=os.getenv("DB_COLLECTION_NAME"),
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name=list(DB_DENSE_INDEX_CONFIG.keys())[0],
    sparse_vector_name=list(DB_SPARSE_INDEX_CONFIG_FR.keys())[0]
)

NB_CHUNKS_RETRIEVED = 100
NB_CHUNKS_SEND_TO_LLM = 5

In [4]:
rag_result = dataset.run_experiment(
    name=os.getenv("DB_COLLECTION_NAME"),
    run_name=f"{RUN_NAME}-rag",
    description=f"Full RAG evaluation on {DATASET_NAME} dataset using {os.getenv("DB_COLLECTION_NAME")} collection",
    task=make_rag_task(qdrant, os.getenv("MODEL_RERANKER_NAME"), os.getenv("MODEL_RERANKER_API_KEY"), os.getenv("MODELS_BASE_URL"), NB_CHUNKS_RETRIEVED, NB_CHUNKS_SEND_TO_LLM),
    evaluators=[
        # retrieval
        make_hit_at_x_evaluator(1),
        make_hit_at_x_evaluator(2),
        make_hit_at_x_evaluator(3),
        make_hit_at_x_evaluator(4),
        make_hit_at_x_evaluator(5),
        make_hit_at_x_evaluator(10),
        make_hit_at_x_evaluator(15),
        make_hit_at_x_evaluator(20),
        mrr_doc_evaluator,
        nb_correct_doc_evaluator,
        # generation
        chrf_evaluator
    ],
    metadata={
        "scope": "retrieval",
        "collection_name": os.getenv("DB_COLLECTION_NAME"),
        "top_k": NB_CHUNKS_RETRIEVED
    }
)

print(rag_result.format())